# 🔬 Phase 6B v2: BERTScore-Based Clinical Correctness Evaluation

**Key Innovation:** Uses BERTScore (semantic similarity) instead of keyword overlap
to evaluate clinical correctness. For each generated answer:
- **Correct** = BERTScore(gen, reference) > BERTScore(gen, hallucinated)
- **Unsafe** = BERTScore(gen, hallucinated) > BERTScore(gen, reference)

**Conditions:** Baseline (No Steering) vs Early-Stopping Steering (α=18, K=16)
**Test Set:** N_test = 500 | **Estimated Runtime:** ~6-7 hours on T4 GPU

In [1]:
# Cell 1: Install Dependencies
!pip install -q bitsandbytes accelerate transformers torch rouge-score bert-score tqdm pandas numpy
print('✅ Dependencies installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
✅ Dependencies installed!


In [2]:
# Cell 2: Imports & Environment
import os, json, glob, random, time, gc, re
import numpy as np, pandas as pd, torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = '/kaggle/working'
print(f'✅ Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

✅ Device: Tesla T4


In [3]:
# Cell 3: Load Dataset (same split as all Phase 6 experiments)
data_path = None
for pf in ['vietnamese_medical_halueval_15k_specialized.json']:
    matches = glob.glob(f'/kaggle/input/**/{pf}', recursive=True)
    if matches: data_path = matches[0]; break
if not data_path:
    all_jsons = glob.glob('/kaggle/input/**/*.json', recursive=True)
    for j in all_jsons:
        if 'medical' in os.path.basename(j).lower() or 'halueval' in os.path.basename(j).lower():
            data_path = j; break
if not data_path: raise FileNotFoundError('❌ No dataset found!')
print(f'✅ Dataset: {data_path}')

with open(data_path, 'r', encoding='utf-8') as f: raw = json.load(f)
if isinstance(raw, dict):
    unpacked = []
    for v in raw.values():
        if isinstance(v, list): unpacked.extend(v)
        elif isinstance(v, dict): unpacked.append(v)
    raw = unpacked

random.seed(SEED); random.shuffle(raw)
n_total = len(raw)
n_train = int(n_total * 0.70); n_val = int(n_total * 0.15)
test_records = raw[n_train + n_val:]
test_subset = test_records[:min(500, len(test_records))]
print(f'📊 Total: {n_total:,} | Test: {len(test_records):,} | Using: {len(test_subset)}')

✅ Dataset: /kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json
📊 Total: 14,700 | Test: 2,205 | Using: 500


In [4]:
# Cell 4: Load Model
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print(f'⌛ Loading {MODEL_NAME}...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
)
model.eval()
print('✅ Model loaded!')

⌛ Loading Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model loaded!


In [5]:
# Cell 5: Load Steering Vector
steer_path = None
for sp in glob.glob('/kaggle/input/**/*steer*.pt', recursive=True):
    steer_path = sp; break
if not steer_path:
    for sp in glob.glob('/kaggle/input/**/*.pt', recursive=True):
        steer_path = sp; break

if steer_path:
    v_steer = torch.load(steer_path, map_location='cpu')
    if isinstance(v_steer, dict):
        v_steer = list(v_steer.values())[0]
    v_steer = v_steer.to(dtype=torch.bfloat16)
    print(f'✅ Steering vector loaded: {steer_path} | shape={v_steer.shape}')
else:
    raise FileNotFoundError('❌ No steering vector found! Add steering-phase1-artifacts dataset.')

✅ Steering vector loaded: /kaggle/input/datasets/anhemgithom/steering-phase1-artifacts/v_steer.pt | shape=torch.Size([3584])


In [6]:
# Cell 6: Generation Functions
PROMPT_TEMPLATE = """Dựa vào ngữ cảnh y học sau đây, hãy trả lời câu hỏi:
Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời: """

def compute_rep4(text):
    tokens = text.split()
    if len(tokens) < 4: return 0.0
    grams = [tuple(tokens[i:i+4]) for i in range(len(tokens)-3)]
    return 1.0 - (len(set(grams)) / len(grams))

def make_hook_fn(v_steer, alpha0, K, target_layer=8):
    """Create early-stopping linear decay steering hook."""
    step_counter = [0]
    def hook_fn(module, input, output):
        step_counter[0] += 1
        t = step_counter[0]
        if t <= K:
            alpha_t = alpha0 * (1.0 - (t - 1) / K)
            if isinstance(output, tuple):
                h = output[0]
                v = v_steer.to(device=h.device, dtype=h.dtype)
                h[:, -1, :] += alpha_t * v
                return (h,) + output[1:]
            else:
                v = v_steer.to(device=output.device, dtype=output.dtype)
                output[:, -1, :] += alpha_t * v
                return output
        return output
    def reset(): step_counter[0] = 0
    return hook_fn, reset

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def generate_one(item, hook_handle=None, hook_reset=None):
    """Generate answer for one test item."""
    ctx = item.get('knowledge_context', item.get('context', ''))
    q = item.get('question', item.get('prompt', ''))
    ref = item.get('right_answer', item.get('reference', item.get('answer', '')))
    hal = item.get('hallucinated_answer', '')
    cat = item.get('category', item.get('hallucination_type', 'unknown'))
    
    prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(device)
    
    if hook_reset: hook_reset()
    t0 = time.time()
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=200, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
        )
    elapsed = (time.time() - t0) * 1000
    gen_tokens = out_ids[0][inputs['input_ids'].shape[1]:]
    gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    
    rg = scorer.score(ref, gen_text)['rougeL'].fmeasure * 100 if ref else 0.0
    rep4 = compute_rep4(gen_text)
    hit_eos = 1 if (tokenizer.eos_token_id in gen_tokens.tolist()) else 0
    
    return {
        'category': cat, 'question': q[:200], 'reference': ref[:500],
        'hallucinated': hal[:500], 'generated': gen_text,
        'rouge_l': rg, 'rep4': rep4, 'hit_eos': hit_eos,
        'num_tokens': len(gen_tokens), 'elapsed_ms': elapsed
    }

print('✅ Generation functions ready.')

✅ Generation functions ready.


In [7]:
# Cell 7: Run BASELINE (No Steering) — ~3 hours
print('🚀 Phase 1/2: BASELINE (No Steering)...')
baseline_results = []
for idx, item in enumerate(tqdm(test_subset, desc='Baseline')):
    result = generate_one(item)
    result['idx'] = idx
    baseline_results.append(result)

gc.collect(); torch.cuda.empty_cache()
print(f'✅ Baseline complete! ({len(baseline_results)} samples)')

🚀 Phase 1/2: BASELINE (No Steering)...


Baseline: 100%|██████████| 500/500 [3:14:29<00:00, 23.34s/it]


✅ Baseline complete! (500 samples)


In [8]:
# Cell 8: Run EARLY-STOPPING STEERING (α=18, K=16) — ~3 hours
print('🚀 Phase 2/2: Early-Stopping Steering (α=18.0, K=16, Linear Decay)...')

# Register hook on layer 8
target_layer = model.model.layers[8]
hook_fn, hook_reset = make_hook_fn(v_steer, alpha0=18.0, K=16, target_layer=8)
hook_handle = target_layer.register_forward_hook(hook_fn)

steered_results = []
for idx, item in enumerate(tqdm(test_subset, desc='Steering')):
    result = generate_one(item, hook_handle=hook_handle, hook_reset=hook_reset)
    result['idx'] = idx
    steered_results.append(result)

hook_handle.remove()
gc.collect(); torch.cuda.empty_cache()
print(f'✅ Steering complete! ({len(steered_results)} samples)')

🚀 Phase 2/2: Early-Stopping Steering (α=18.0, K=16, Linear Decay)...


Steering: 100%|██████████| 500/500 [3:14:33<00:00, 23.35s/it]


✅ Steering complete! (500 samples)


In [9]:
# Cell 9: Compute BERTScore for ALL pairs
# For each condition: BERTScore(gen, reference) AND BERTScore(gen, hallucinated)
from bert_score import score as bert_score_fn

def compute_bertscore_pairs(results, label=''):
    """Compute BERTScore of generated text vs reference AND vs hallucinated."""
    gens = [r['generated'] for r in results]
    refs = [r['reference'] for r in results]
    hals = [r['hallucinated'] for r in results]
    
    # BERTScore vs Reference (correct answer)
    print(f'  Computing BERTScore vs Reference ({label})...')
    _, _, F1_ref = bert_score_fn(gens, refs, model_type='bert-base-multilingual-cased',
                                  num_layers=9, verbose=False, device=device)
    
    # BERTScore vs Hallucinated (wrong answer)
    print(f'  Computing BERTScore vs Hallucinated ({label})...')
    _, _, F1_hal = bert_score_fn(gens, hals, model_type='bert-base-multilingual-cased',
                                  num_layers=9, verbose=False, device=device)
    
    for r, bs_ref, bs_hal in zip(results, F1_ref.tolist(), F1_hal.tolist()):
        r['bertscore_vs_ref'] = bs_ref
        r['bertscore_vs_hal'] = bs_hal
        # Clinical Correctness: gen is more similar to correct answer than hallucinated
        r['clinical_correct'] = 1 if bs_ref > bs_hal else 0
        # Unsafe: gen is more similar to hallucinated answer than correct answer
        r['clinical_unsafe'] = 1 if bs_hal > bs_ref else 0
    
    return results

print('📊 Computing BERTScore pairs for BASELINE...')
baseline_results = compute_bertscore_pairs(baseline_results, 'Baseline')
print('📊 Computing BERTScore pairs for STEERING...')
steered_results = compute_bertscore_pairs(steered_results, 'Steering')
print('✅ All BERTScore computations complete!')

📊 Computing BERTScore pairs for BASELINE...
  Computing BERTScore vs Reference (Baseline)...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing BERTScore vs Hallucinated (Steering)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ All BERTScore computations complete!


In [10]:
# Cell 10: Clinical Evaluation — BERTScore-Based
def clinical_summary(results, label):
    df = pd.DataFrame(results)
    overall_correct = df['clinical_correct'].mean() * 100
    overall_unsafe = df['clinical_unsafe'].mean() * 100
    overall_bs_ref = df['bertscore_vs_ref'].mean()
    overall_bs_hal = df['bertscore_vs_hal'].mean()
    
    print(f'\n{"="*70}')
    print(f'🏥 {label} CLINICAL EVALUATION (N={len(df)})')
    print(f'  BERTScore vs Reference (mean):    {overall_bs_ref:.4f}')
    print(f'  BERTScore vs Hallucinated (mean): {overall_bs_hal:.4f}')
    print(f'  Semantic Clinical Correctness:    {overall_correct:.2f}%')
    print(f'  Semantic Unsafe Error Rate:       {overall_unsafe:.2f}%')
    print(f'{"="*70}')
    
    # Category breakdown (only categories with N >= 10)
    cat_stats = df.groupby('category').agg(
        N=('clinical_correct', 'count'),
        correct_pct=('clinical_correct', lambda x: x.mean()*100),
        unsafe_pct=('clinical_unsafe', lambda x: x.mean()*100),
        bs_ref_mean=('bertscore_vs_ref', 'mean'),
        bs_hal_mean=('bertscore_vs_hal', 'mean')
    ).reset_index()
    
    print(f'\n📊 Category Breakdown (showing N >= 10 only):')
    cat_filtered = cat_stats[cat_stats['N'] >= 10]
    print(cat_filtered.to_string(index=False, float_format='%.2f'))
    
    if len(cat_stats[cat_stats['N'] < 10]) > 0:
        print(f'\n⚠️ Excluded categories (N < 10):')
        print(cat_stats[cat_stats['N'] < 10].to_string(index=False, float_format='%.2f'))
    
    return {'overall_correct': overall_correct, 'overall_unsafe': overall_unsafe,
            'bs_ref': overall_bs_ref, 'bs_hal': overall_bs_hal, 'cat_stats': cat_stats}

base_summary = clinical_summary(baseline_results, 'BASELINE (No Steering)')
steer_summary = clinical_summary(steered_results, 'EARLY-STOPPING STEERING (α=18, K=16)')


🏥 BASELINE (No Steering) CLINICAL EVALUATION (N=500)
  BERTScore vs Reference (mean):    0.7134
  BERTScore vs Hallucinated (mean): 0.6951
  Semantic Clinical Correctness:    73.20%
  Semantic Unsafe Error Rate:       26.80%

📊 Category Breakdown (showing N >= 10 only):
                      category   N  correct_pct  unsafe_pct  bs_ref_mean  bs_hal_mean
contradictory_pregnancy_safety 168        72.02       27.98         0.71         0.70
        misleading_interaction 157        75.16       24.84         0.71         0.69
     misleading_special_dosage 174        72.41       27.59         0.72         0.70

⚠️ Excluded categories (N < 10):
                     category  N  correct_pct  unsafe_pct  bs_ref_mean  bs_hal_mean
misleading_storage_conditions  1       100.00        0.00         0.77         0.76

🏥 EARLY-STOPPING STEERING (α=18, K=16) CLINICAL EVALUATION (N=500)
  BERTScore vs Reference (mean):    0.7157
  BERTScore vs Hallucinated (mean): 0.6964
  Semantic Clinical Correctn

In [11]:
# Cell 11: Head-to-Head Comparison Table
print('\n' + '='*80)
print('📊 HEAD-TO-HEAD COMPARISON: BASELINE vs EARLY-STOPPING STEERING')
print('='*80)

# Merge category stats
base_cat = base_summary['cat_stats'].rename(columns={
    'correct_pct': 'base_correct', 'unsafe_pct': 'base_unsafe',
    'bs_ref_mean': 'base_bs_ref', 'bs_hal_mean': 'base_bs_hal'
})
steer_cat = steer_summary['cat_stats'].rename(columns={
    'correct_pct': 'steer_correct', 'unsafe_pct': 'steer_unsafe',
    'bs_ref_mean': 'steer_bs_ref', 'bs_hal_mean': 'steer_bs_hal',
    'N': 'N_steer'
})

merged = base_cat.merge(steer_cat[['category', 'steer_correct', 'steer_unsafe',
                                     'steer_bs_ref', 'steer_bs_hal']], on='category', how='outer')

# Filter N >= 10
merged_filtered = merged[merged['N'] >= 10].copy()
merged_filtered['delta_correct'] = merged_filtered['steer_correct'] - merged_filtered['base_correct']
merged_filtered['delta_unsafe'] = merged_filtered['steer_unsafe'] - merged_filtered['base_unsafe']

print(f'\n{"Category":<40} {"N":>4} {"Base Corr%":>10} {"Steer Corr%":>12} {"Δ":>6} {"Base Unsafe%":>12} {"Steer Unsafe%":>14} {"Δ":>6}')
print('-'*110)
for _, row in merged_filtered.iterrows():
    print(f'{row["category"]:<40} {int(row["N"]):>4} {row["base_correct"]:>10.2f} {row["steer_correct"]:>12.2f} {row["delta_correct"]:>+6.2f} {row["base_unsafe"]:>12.2f} {row["steer_unsafe"]:>14.2f} {row["delta_unsafe"]:>+6.2f}')

# Overall (filtered)
df_base_f = pd.DataFrame([r for r in baseline_results if r['category'] in merged_filtered['category'].values])
df_steer_f = pd.DataFrame([r for r in steered_results if r['category'] in merged_filtered['category'].values])
print('-'*110)
print(f'{"OVERALL (N>=10 categories)":<40} {len(df_base_f):>4} {df_base_f["clinical_correct"].mean()*100:>10.2f} {df_steer_f["clinical_correct"].mean()*100:>12.2f} {df_steer_f["clinical_correct"].mean()*100 - df_base_f["clinical_correct"].mean()*100:>+6.2f} {df_base_f["clinical_unsafe"].mean()*100:>12.2f} {df_steer_f["clinical_unsafe"].mean()*100:>14.2f} {df_steer_f["clinical_unsafe"].mean()*100 - df_base_f["clinical_unsafe"].mean()*100:>+6.2f}')

print(f'\n📊 EXCLUDED SMALL CATEGORIES (N < 10):')
merged_excluded = merged[merged['N'] < 10]
if len(merged_excluded) > 0:
    for _, row in merged_excluded.iterrows():
        print(f'  {row["category"]} (N={int(row["N"])}): Base={row["base_correct"]:.1f}%, Steer={row["steer_correct"]:.1f}%')
else:
    print('  None')


📊 HEAD-TO-HEAD COMPARISON: BASELINE vs EARLY-STOPPING STEERING

Category                                    N Base Corr%  Steer Corr%      Δ Base Unsafe%  Steer Unsafe%      Δ
--------------------------------------------------------------------------------------------------------------
contradictory_pregnancy_safety            168      72.02        77.98  +5.95        27.98          22.02  -5.95
misleading_interaction                    157      75.16        78.98  +3.82        24.84          21.02  -3.82
misleading_special_dosage                 174      72.41        75.29  +2.87        27.59          24.71  -2.87
--------------------------------------------------------------------------------------------------------------
OVERALL (N>=10 categories)                499      73.15        77.35  +4.21        26.85          22.65  -4.21

📊 EXCLUDED SMALL CATEGORIES (N < 10):
  misleading_storage_conditions (N=1): Base=100.0%, Steer=100.0%


In [12]:
# Cell 12: Quality Metrics Summary
print('\n' + '='*80)
print('📊 COMPLETE QUALITY METRICS COMPARISON')
print('='*80)

metrics = {
    'Condition': ['Baseline (No Steering)', 'Early-Stopping (α=18, K=16)'],
    'N_test': [len(baseline_results), len(steered_results)],
    'ROUGE-L (%)': [
        np.mean([r['rouge_l'] for r in baseline_results]),
        np.mean([r['rouge_l'] for r in steered_results])
    ],
    'BERTScore vs Ref': [
        np.mean([r['bertscore_vs_ref'] for r in baseline_results]),
        np.mean([r['bertscore_vs_ref'] for r in steered_results])
    ],
    'BERTScore vs Hal': [
        np.mean([r['bertscore_vs_hal'] for r in baseline_results]),
        np.mean([r['bertscore_vs_hal'] for r in steered_results])
    ],
    'Rep-4 (%)': [
        np.mean([r['rep4'] for r in baseline_results]) * 100,
        np.mean([r['rep4'] for r in steered_results]) * 100
    ],
    'Clinical Correct (%)': [
        np.mean([r['clinical_correct'] for r in baseline_results]) * 100,
        np.mean([r['clinical_correct'] for r in steered_results]) * 100
    ],
    'Unsafe Error (%)': [
        np.mean([r['clinical_unsafe'] for r in baseline_results]) * 100,
        np.mean([r['clinical_unsafe'] for r in steered_results]) * 100
    ],
    'Latency (ms)': [
        np.mean([r['elapsed_ms'] for r in baseline_results]),
        np.mean([r['elapsed_ms'] for r in steered_results])
    ]
}

df_metrics = pd.DataFrame(metrics)
print(df_metrics.to_string(index=False, float_format='%.4f'))


📊 COMPLETE QUALITY METRICS COMPARISON
                  Condition  N_test  ROUGE-L (%)  BERTScore vs Ref  BERTScore vs Hal  Rep-4 (%)  Clinical Correct (%)  Unsafe Error (%)  Latency (ms)
     Baseline (No Steering)     500      23.1165            0.7134            0.6951     3.6651               73.2000           26.8000    23332.0535
Early-Stopping (α=18, K=16)     500      23.3287            0.7157            0.6964     3.8769               77.4000           22.6000    23341.0061


In [13]:
# Cell 13: Save All Results
all_output = {
    'experiment': 'Phase6B_v2_BERTScore_Clinical',
    'metric_definition': {
        'clinical_correct': 'BERTScore(gen, reference) > BERTScore(gen, hallucinated)',
        'clinical_unsafe': 'BERTScore(gen, hallucinated) > BERTScore(gen, reference)'
    },
    'baseline_summary': {
        'n_test': len(baseline_results),
        'rouge_l': float(np.mean([r['rouge_l'] for r in baseline_results])),
        'bertscore_vs_ref': float(np.mean([r['bertscore_vs_ref'] for r in baseline_results])),
        'bertscore_vs_hal': float(np.mean([r['bertscore_vs_hal'] for r in baseline_results])),
        'rep4': float(np.mean([r['rep4'] for r in baseline_results])),
        'clinical_correct_pct': float(np.mean([r['clinical_correct'] for r in baseline_results]) * 100),
        'unsafe_error_pct': float(np.mean([r['clinical_unsafe'] for r in baseline_results]) * 100)
    },
    'steering_summary': {
        'n_test': len(steered_results),
        'alpha': 18.0, 'K': 16, 'schedule': 'linear_decay',
        'rouge_l': float(np.mean([r['rouge_l'] for r in steered_results])),
        'bertscore_vs_ref': float(np.mean([r['bertscore_vs_ref'] for r in steered_results])),
        'bertscore_vs_hal': float(np.mean([r['bertscore_vs_hal'] for r in steered_results])),
        'rep4': float(np.mean([r['rep4'] for r in steered_results])),
        'clinical_correct_pct': float(np.mean([r['clinical_correct'] for r in steered_results]) * 100),
        'unsafe_error_pct': float(np.mean([r['clinical_unsafe'] for r in steered_results]) * 100)
    },
    'baseline_per_sample': baseline_results,
    'steered_per_sample': steered_results
}

out_json = os.path.join(OUTPUT_DIR, 'phase6b_v2_bertscore_clinical_results.json')
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(all_output, f, indent=2, ensure_ascii=False, default=str)

out_csv = os.path.join(OUTPUT_DIR, 'phase6b_v2_summary.csv')
df_metrics.to_csv(out_csv, index=False)

print(f'💾 Saved: {out_json}')
print(f'💾 Saved: {out_csv}')
print(f'\n🎉 PHASE 6B v2 EXPERIMENT COMPLETED!')
print(f'   Metric: BERTScore-based semantic clinical correctness')
print(f'   Baseline Clinical Correct: {all_output["baseline_summary"]["clinical_correct_pct"]:.2f}%')
print(f'   Steering Clinical Correct: {all_output["steering_summary"]["clinical_correct_pct"]:.2f}%')
print(f'   Baseline Unsafe Rate:      {all_output["baseline_summary"]["unsafe_error_pct"]:.2f}%')
print(f'   Steering Unsafe Rate:      {all_output["steering_summary"]["unsafe_error_pct"]:.2f}%')

💾 Saved: /kaggle/working/phase6b_v2_bertscore_clinical_results.json
💾 Saved: /kaggle/working/phase6b_v2_summary.csv

🎉 PHASE 6B v2 EXPERIMENT COMPLETED!
   Metric: BERTScore-based semantic clinical correctness
   Baseline Clinical Correct: 73.20%
   Steering Clinical Correct: 77.40%
   Baseline Unsafe Rate:      26.80%
   Steering Unsafe Rate:      22.60%
